# KG Unification — Phase 2, Step 5: Entity Normalization

Merges MeSH, CogAtlas, and NLP knowledge graphs into a single canonical entity table.

### Merge priority
1. **MeSH** — DescriptorUI is the canonical ID for any concept that exists in MeSH
2. **CogAtlas** — string-match against MeSH synonyms; unmapped concepts get `cogat_{trm_id}`
3. **NLP** — type-conditional strategy:
   - `modality / method / metric` — exact match only; discard on no match (noisy types)
   - `disorder / anatomical_region / cognitive_construct / network / intervention` — exact match
     → word-boundary substring match → keep as `nlp_{slug}` if still unmatched

NLP's value is primarily in its *edges*: a co_occurs_with edge between two NLP terms that both
map to MeSH nodes survives in the unified graph under the canonical MeSH IDs.

### Outputs (`data/unified_kg/`)
| file | description |
|---|---|
| `unified_kg_nodes.parquet` | Master entity table — one row per canonical concept |
| `unified_kg_edges.parquet` | All edges from all three KGs, endpoints re-pointed to canonical IDs |
| `cogat_kg_nodes.parquet` | CogAtlas concepts with match metadata |
| `merge_log.json` | Full audit log — one record per mapped/discarded entity |

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
import logging

# Paths are relative to experiments/
MESH_DIR  = Path("data/mesh_kg")
NLP_DIR   = Path("data/nlp_kg")
COGAT_DIR = Path("data/cogatlas")
OUT_DIR   = Path("data/unified_kg")

sys.path.insert(0, str(Path("..").resolve() / "src"))

from neurovlm.gnn.normalize import (
    KEEP_TYPES,
    NLP_NODE_FILTERS,   # mutable dict — extend in-place to add domain-specific filters
    build_mesh_synonym_lookup,
    fetch_cogatlas_concepts,
    fetch_cogatlas_relations,
    build_cogatlas_kg,
    normalize_nlp_entities,
    build_master_entity_table,
    remap_edges,
    run_normalization,
    _norm,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

## Load input KGs

In [2]:
mesh_descriptors = pd.read_parquet(MESH_DIR / "mesh_descriptors.parquet")
mesh_nodes       = pd.read_parquet(MESH_DIR / "mesh_kg_nodes.parquet")
mesh_edges       = pd.read_parquet(MESH_DIR / "mesh_kg_edges_all.parquet")
nlp_nodes        = pd.read_parquet(NLP_DIR  / "nlp_kg_nodes.parquet")
nlp_edges        = pd.read_parquet(NLP_DIR  / "nlp_kg_edges.parquet")

print(f"MeSH descriptors : {len(mesh_descriptors):>7,}")
print(f"MeSH nodes       : {len(mesh_nodes):>7,}")
print(f"MeSH edges       : {len(mesh_edges):>7,}")
print(f"NLP nodes        : {len(nlp_nodes):>7,}")
print(f"NLP edges        : {len(nlp_edges):>7,}")

MeSH descriptors :  31,110
MeSH nodes       :  31,110
MeSH edges       : 305,227
NLP nodes        :  12,841
NLP edges        : 155,352


## Step 1 — MeSH synonym lookup

In [3]:
mesh_lookup = build_mesh_synonym_lookup(mesh_descriptors)
print(f"MeSH synonym lookup entries: {len(mesh_lookup):,}")

test_terms = ["motor cortex", "working memory", "fmri", "hippocampus", "alzheimer disease"]
print("\nSpot-check exact lookups:")
for t in test_terms:
    ui = mesh_lookup.get(_norm(t), "NOT FOUND")
    name = mesh_descriptors.set_index("ui")["name"].get(ui, "") if ui != "NOT FOUND" else ""
    print(f"  {t!r:30s} → {ui}  {name}")

INFO neurovlm.gnn.normalize: MeSH synonym lookup: 245494 entries for 31110 descriptors (21528 collisions ignored)


MeSH synonym lookup entries: 245,494

Spot-check exact lookups:
  'motor cortex'                 → D009044  Motor Cortex
  'working memory'               → D008570  Memory, Short-Term
  'fmri'                         → D008279  Magnetic Resonance Imaging
  'hippocampus'                  → D006624  Hippocampus
  'alzheimer disease'            → D000544  Alzheimer Disease


## Step 2 — CogAtlas KG

Fetches concept IDs and names from cognitiveatlas.org (cached after first run).
Maps each concept to a canonical ID via MeSH string matching; unmapped → `cogat_{trm_id}`.

In [4]:
COGAT_DIR.mkdir(parents=True, exist_ok=True)
cogatlas_df = fetch_cogatlas_concepts(out_path=COGAT_DIR / "cogatlas_concepts.parquet")
print(f"CogAtlas concepts: {len(cogatlas_df):,}")

# Scrape is-a-kind-of relations from individual concept pages (cached after first run)
cogat_relations_df = fetch_cogatlas_relations(
    cogatlas_df,
    out_path=COGAT_DIR / "cogatlas_relations.parquet",
)
print(f"CogAtlas kind_of relations: {len(cogat_relations_df):,}")
cogat_relations_df.head(5)

INFO neurovlm.gnn.normalize: Loading CogAtlas concepts from data/cogatlas/cogatlas_concepts.parquet


CogAtlas concepts: 918


,trm_id,name
0,trm_4a3fd79d096be,abductive reasoning
1,trm_4a3fd79d096e3,abstract analogy
2,trm_4a3fd79d096f0,abstract knowledge
3,trm_4a3fd79d096fc,acoustic coding
4,trm_4a3fd79d09707,acoustic encoding


In [5]:
cogat_nodes, cogat_edges, cogat_to_canonical = build_cogatlas_kg(
    cogatlas_df,
    mesh_lookup,
    cogatlas_edges_df=cogat_relations_df,
)
print(cogat_nodes["match_type"].value_counts().to_string())
print(f"
CogAtlas edges built: {len(cogat_edges):,}")

mesh_matched = cogat_nodes[cogat_nodes["match_type"] == "mesh_synonym_match"]
new_cogat    = cogat_nodes[cogat_nodes["match_type"] == "new_cogat_node"]
print("
=== Sample CogAtlas → MeSH matches ===")
print(mesh_matched[["name", "canonical_id"]].head(12).to_string(index=False))
print("
=== Sample new cogat_ nodes ===")
print(new_cogat[["name", "canonical_id"]].head(8).to_string(index=False))

INFO neurovlm.gnn.normalize: CogAtlas: 918 concepts → 166 mapped to MeSH, 752 new cogat_ nodes


match_type
new_cogat_node        752
mesh_synonym_match    166

=== Sample CogAtlas → MeSH matches ===
                  name canonical_id
              altruism      D000533
             anhedonia      D059445
antisocial personality      D000987
               anxiety      D001007
              appetite      D001066
               arousal      D001143
           association      D001244
  association learning      D001245
             attention      D001288
      attentional bias   D000070379
     attentional blink      D054518
              attitude      D001290

=== Sample new cogat_ nodes ===
                        name            canonical_id
         abductive reasoning cogat_trm_4a3fd79d096be
            abstract analogy cogat_trm_4a3fd79d096e3
          abstract knowledge cogat_trm_4a3fd79d096f0
             acoustic coding cogat_trm_4a3fd79d096fc
           acoustic encoding cogat_trm_4a3fd79d09707
acoustic phonetic processing cogat_trm_4a3fd79d09713
         acoustic process

## Step 3 — NLP entity normalization

Type-conditional strategy (see `normalize.KEEP_TYPES`):
- **`modality / method / metric`** — exact match only; discard on no match
- **`disorder / anatomical_region / cognitive_construct / network / intervention`** —
  exact match → word-boundary substring match → keep as `nlp_` node if still no match

Substring matching prefers primary descriptor names over synonyms (avoids e.g.
"alzheimer" resolving to Amyloid beta-Peptides instead of Alzheimer Disease).

Known ambiguous matches can be overridden in `FORCE_MAP` below.

In [6]:
# FORCE_MAP: NLP term → canonical_id override.
# Use this to correct known-bad substring matches discovered during audit.
# Example entries are commented out — uncomment and extend as needed.
FORCE_MAP: dict[str, str] = {
    # "occipital": "D009778",          # Occipital Lobe (beats shorter Occipital Bone)
    # "nucleus basalis": "D020532",    # Basal Nucleus of Meynert (beats Olivary Nucleus)
    # "hippocampal": "D006624",        # Hippocampus (beats Hippocampal Sclerosis)
    # "subcortical": "nlp_subcortical", # keep as nlp_ rather than Cerebral Infarction
}

In [7]:
import re

# NLP_NODE_FILTERS is a mutable module-level dict — extend it here to add
# domain-specific secondary discard rules for KEEP_TYPE nodes that survive
# all matching attempts but are still wrong (e.g. classifier errors).
#
# The default anatomical_region filter already handles the most common
# misclassifications (functional states "brain activity/activation", modality
# contamination "brain mri", over-generic standalone terms "hemisphere/lobes").
#
# Examples of additional rules (uncomment to activate):
# NLP_NODE_FILTERS["disorder"] = re.compile(
#     r"\b(patients?|surgery|uptake)\b"  # population or procedure terms mislabelled as disorder
# )
# NLP_NODE_FILTERS["intervention"] = re.compile(
#     r"\boptions?\b"  # "treatment option/options" — generic clinical management
# )
print("Active NLP_NODE_FILTERS:")
for ntype, pat in NLP_NODE_FILTERS.items():
    print(f"  {ntype}: {pat.pattern}")

Active NLP_NODE_FILTERS:
  anatomical_region: \b(?:activity|activation)\b|\bmri\b|^(?:hemisphere|lobes|whole-brain|whole brain)$


In [8]:
# Build the primary-name set so substring matching prefers primary names over synonyms
mesh_primary_names = frozenset(_norm(str(r["name"])) for _, r in mesh_descriptors.iterrows())

cogat_name_lookup = {
    _norm(str(row["name"])): str(row["canonical_id"])
    for _, row in cogat_nodes.iterrows()
    if _norm(str(row["name"]))
}

nlp_to_canonical, nlp_merge_log = normalize_nlp_entities(
    nlp_nodes,
    mesh_lookup,
    cogat_name_lookup,
    keep_unmatched=False,
    keep_types=KEEP_TYPES,          # disorder, anatomical_region, cognitive_construct, network, intervention
    min_substring_len=6,
    mesh_primary_names=mesh_primary_names,
)

# Apply FORCE_MAP overrides
nlp_to_canonical.update(FORCE_MAP)

log_df = pd.DataFrame(nlp_merge_log)
print(log_df["match_type"].value_counts().to_string())
print(f"\nNLP terms with a canonical ID: {len(nlp_to_canonical):,} / {len(nlp_nodes):,}")

INFO neurovlm.gnn.normalize: NLP normalization (12841 terms): 1662 exact MeSH, 543 substring MeSH, 73 CogAtlas, 3296 new nlp_ nodes, 7267 discarded


match_type
discarded               7267
new_nlp_node            3296
mesh_synonym_match      1662
mesh_substring_match     543
cogat_name_match          73

NLP terms with a canonical ID: 5,574 / 12,841


In [9]:
print("Match breakdown per node_type:")
print(
    log_df.groupby(["node_type", "match_type"])
    .size()
    .unstack(fill_value=0)
    .to_string()
)

Match breakdown per node_type:
match_type           cogat_name_match  discarded  mesh_substring_match  mesh_synonym_match  new_nlp_node
node_type                                                                                               
anatomical_region                   2         47                   211                 220          1303
cognitive_construct                50          0                    89                  86           483
disorder                            0          0                   160                 389           848
intervention                        0          0                    73                 130           474
method                              4       1662                     0                 167             0
metric                              5       1350                     0                 115             0
modality                           12       4208                     0                 539             0
network                 

In [10]:
# Inspect all substring matches — scan for false positives and add to FORCE_MAP above
ui_to_name = mesh_descriptors.set_index("ui")["name"]
substr = log_df[log_df["match_type"] == "mesh_substring_match"].copy()
substr["mesh_name"] = substr["canonical_id"].map(ui_to_name)

print(f"Total substring matches: {len(substr)}")
print()
for ntype in ["disorder", "anatomical_region", "cognitive_construct", "network", "intervention"]:
    sub = substr[substr["node_type"] == ntype][["entity", "mesh_name"]]
    if len(sub) > 0:
        print(f"--- {ntype} ({len(sub)}) ---")
        print(sub.to_string(index=False))
        print()

Total substring matches: 543

--- disorder (160) ---
                                  entity                                    mesh_name
                   absence epilepsy rats                            Epilepsy, Absence
                   acute stroke patients                                       Stroke
                  acute stroke treatment                                       Stroke
                                  alarms                              Clinical Alarms
                               alzheimer                            Alzheimer Disease
          ankylosing spondylitis disease                      Spondylitis, Ankylosing
                       antibody syndrome                      Autoimmune Hypophysitis
                          apnea syndrome                        Sleep Apnea Syndromes
                        arterial disease               Intracranial Arterial Diseases
                        artery aneurysms                        Intracranial Aneurysm
 

### LLM audit of substring matches

For each NLP→MeSH substring match, asks the LLM whether the mapping is semantically correct.
False positives should be added to `FORCE_MAP` (override to correct MeSH ID or `nlp_` slug).

In [11]:
import re
import ollama
from tqdm.auto import tqdm

OLLAMA_MODEL = "qwen2.5:7b-instruct"
SUBSTR_BATCH_SIZE = 15

# Build the check list from the substring match audit above
ui_to_name_map = mesh_descriptors.set_index("ui")["name"]
substr_check = log_df[log_df["match_type"] == "mesh_substring_match"].copy()
substr_check["mesh_name"] = substr_check["canonical_id"].map(ui_to_name_map)
# Merge degree info
substr_check = substr_check.merge(
    nlp_nodes[["node_id", "degree"]], left_on="entity", right_on="node_id", how="left"
).sort_values("degree", ascending=False).reset_index(drop=True)

SUBSTR_SYSTEM = (
    "You are an expert in neuroscience. "
    "For each numbered item you are given an NLP term extracted from neuroscience papers "
    "and a MeSH concept it was matched to via substring matching. "
    "Decide only whether the MeSH concept is the correct or best match for the NLP term — "
    "ignore what category the term belongs to. "
    "Reply with exactly one line per item using the format:\n"
    "N. YES|NO|UNCERTAIN — <reason in ≤8 words>\n"
    "Do not add any other text."
)

def _make_substr_prompt(batch: list[dict]) -> str:
    return "\n".join(
        f'{i+1}. nlp="{r["entity"]}"  mesh="{r["mesh_name"]}"'
        for i, r in enumerate(batch)
    )

def _parse_substr_response(text: str, batch: list[dict]) -> list[dict]:
    flagged = []
    for line in text.strip().splitlines():
        line = line.strip()
        m = re.match(r"^(\d+)\.\s*(YES|NO|UNCERTAIN)[^\w]*(.*)$", line, re.IGNORECASE)
        if not m:
            continue
        idx = int(m.group(1)) - 1
        verdict = m.group(2).upper()
        reason = m.group(3).strip(" —-")
        if 0 <= idx < len(batch) and verdict in ("NO", "UNCERTAIN"):
            flagged.append({
                "node_id": batch[idx]["entity"],
                "mesh_name": batch[idx]["mesh_name"],
                "canonical_id": batch[idx]["canonical_id"],
                "node_type": batch[idx]["node_type"],
                "degree": batch[idx].get("degree"),
                "verdict": verdict,
                "reason": reason,
            })
    return flagged

# ── Run ──────────────────────────────────────────────────────────────────────
records = substr_check.to_dict("records")
batches = [records[i : i + SUBSTR_BATCH_SIZE] for i in range(0, len(records), SUBSTR_BATCH_SIZE)]

print(f"Checking {len(records):,} substring matches "
      f"({len(batches)} batches × {SUBSTR_BATCH_SIZE})  model={OLLAMA_MODEL}")

substr_flagged: list[dict] = []
substr_errors: list[str] = []

for batch in tqdm(batches):
    try:
        resp = ollama.chat(
            model=OLLAMA_MODEL,
            messages=[
                {"role": "system", "content": SUBSTR_SYSTEM},
                {"role": "user",   "content": _make_substr_prompt(batch)},
            ],
            options={"temperature": 0},
        )
        substr_flagged.extend(_parse_substr_response(resp["message"]["content"], batch))
    except Exception as exc:
        substr_errors.append(f"{batch[0]['entity']}: {exc}")

# ── Results ──────────────────────────────────────────────────────────────────
substr_flagged_df = (
    pd.DataFrame(substr_flagged)
    if substr_flagged
    else pd.DataFrame(columns=["node_id", "mesh_name", "canonical_id", "node_type", "degree", "verdict", "reason"])
)

print(f"\nFlagged {len(substr_flagged_df)} / {len(records)} substring matches "
      f"({len(substr_errors)} batch errors)")
if substr_errors:
    print("Errors:", substr_errors[:5])

print()
print("Add these to FORCE_MAP above with the correct canonical_id (or 'nlp_<term>'):")
print(substr_flagged_df.sort_values("degree", ascending=False)
      [["node_id", "mesh_name", "node_type", "degree", "verdict", "reason"]]
      .to_string(index=False))


Checking 543 substring matches (37 batches × 15)  model=qwen2.5:7b-instruct


  0%|          | 0/37 [00:00<?, ?it/s]

INFO httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: PO


Flagged 243 / 543 substring matches (0 batch errors)

Add these to FORCE_MAP above with the correct canonical_id (or 'nlp_<term>'):
                           node_id                                          mesh_name           node_type  degree   verdict                                                                                    reason
                         occipital                                     Occipital Bone   anatomical_region    6632        NO                                                                     wrong part of anatomy
                     interventions                        Internet-Based Intervention        intervention    6603        NO                                                                         different concept
                       subcortical                                Cerebral Infarction   anatomical_region    5784        NO                                                                       incorrect condition
           

In [14]:
# Fix wrong substring matches: remap flagged KEEP_TYPE nodes to nlp_<slug>
# instead of the incorrect MeSH ID. DISCARD_TYPE nodes are dropped entirely.
# Re-run from "Step 4 — Master entity table" onward after this cell.

import re as _re

n_remapped = 0
n_dropped = 0
for _, row in substr_flagged_df.iterrows():
    nid = row["node_id"]
    if nid not in nlp_to_canonical:
        continue  # already removed by earlier step
    if row["node_type"] in KEEP_TYPES:
        slug = _re.sub(r"\s+", "_", nid)[:64]
        nlp_to_canonical[nid] = f"nlp_{slug}"
        n_remapped += 1
    else:
        del nlp_to_canonical[nid]
        n_dropped += 1

print(f"Remapped to nlp_ : {n_remapped:,}  (KEEP_TYPES nodes with wrong MeSH)")
print(f"Dropped           : {n_dropped:,}  (DISCARD_TYPES nodes with wrong MeSH)")
print(f"Remaining mapping : {len(nlp_to_canonical):,}")
print()
print("Re-run from 'Step 4 — Master entity table' to rebuild with updated mapping.")


Remapped to nlp_ : 243  (KEEP_TYPES nodes with wrong MeSH)
Dropped           : 0  (DISCARD_TYPES nodes with wrong MeSH)
Remaining mapping : 5,574

Re-run from 'Step 4 — Master entity table' to rebuild with updated mapping.


In [15]:
# Inspect nlp_ nodes kept from KEEP_TYPES — these are real concepts without MeSH mapping
kept_nlp = log_df[log_df["match_type"] == "new_nlp_node"].copy()
kept_nlp_nodes = nlp_nodes[nlp_nodes["node_id"].isin(kept_nlp["entity"])]

print(f"nlp_ nodes by type:")
print(kept_nlp["node_type"].value_counts().to_string())
print()
print("Top 20 nlp_ nodes by degree (highest graph coverage):")
print(
    kept_nlp_nodes.nlargest(20, "degree")[["node_id", "node_type", "degree"]]
    .to_string(index=False)
)

nlp_ nodes by type:
node_type
anatomical_region      1303
disorder                848
cognitive_construct     483
intervention            474
network                 188

Top 20 nlp_ nodes by degree (highest graph coverage):
                 node_id           node_type  degree
           brain network             network    3201
            mode network             network    3008
         stroke patients            disorder    2804
         medial temporal   anatomical_region    2739
       medial prefrontal   anatomical_region    2736
        cortical regions   anatomical_region    2592
       treatment options        intervention    2272
        treatment option        intervention    2147
        brain functional   anatomical_region    1881
                     dmn             network    1860
     functional networks             network    1832
          frontoparietal   anatomical_region    1806
  endovascular treatment        intervention    1742
              unaffected cognitiv

### LLM audit of kept nlp_ nodes

Uses Ollama (`qwen2.5:7b-instruct`) to flag nodes where the concept name looks wrong for its entity type.
Nodes are checked in degree-descending order so high-impact mismatches surface first.
Results accumulate in `flagged_df` — add confirmed bad matches to `FORCE_MAP` or `NLP_NODE_FILTERS`.

In [16]:
import re
import ollama
from tqdm.auto import tqdm

OLLAMA_MODEL = "qwen2.5:7b-instruct"
BATCH_SIZE = 20   # nodes per LLM call — larger batches are faster but harder to parse

# Sort by degree desc so the most-connected (highest-impact) nodes are verified first
nodes_to_check = (
    kept_nlp_nodes[["node_id", "node_type", "degree"]]
    .sort_values("degree", ascending=False)
    .reset_index(drop=True)
)

SYSTEM_PROMPT = (
    "You are an expert in neuroscience ontologies. "
    "For each numbered item, decide whether the concept name is a valid instance "
    "of the entity type in a neuroscience knowledge graph. "
    "Reply with exactly one line per item using the format:\n"
    "N. YES|NO|UNCERTAIN — <reason in ≤8 words>\n"
    "Do not add any other text."
)

def _make_user_prompt(batch: list[dict]) -> str:
    return "\n".join(
        f'{i+1}. concept="{r["node_id"]}"  type="{r["node_type"]}"'
        for i, r in enumerate(batch)
    )

def _parse_response(text: str, batch: list[dict]) -> list[dict]:
    flagged = []
    for line in text.strip().splitlines():
        line = line.strip()
        m = re.match(r"^(\d+)\.\s*(YES|NO|UNCERTAIN)[^\w]*(.*)$", line, re.IGNORECASE)
        if not m:
            continue
        idx = int(m.group(1)) - 1
        verdict = m.group(2).upper()
        reason = m.group(3).strip(" —-")
        if 0 <= idx < len(batch) and verdict in ("NO", "UNCERTAIN"):
            flagged.append({**batch[idx], "verdict": verdict, "reason": reason})
    return flagged

# ── Run ──────────────────────────────────────────────────────────────────────
batches = [
    nodes_to_check.iloc[i : i + BATCH_SIZE].to_dict("records")
    for i in range(0, len(nodes_to_check), BATCH_SIZE)
]

print(f"Checking {len(nodes_to_check):,} nlp_ nodes "
      f"({len(batches)} batches × {BATCH_SIZE})  model={OLLAMA_MODEL}")

flagged_nodes: list[dict] = []
errors: list[str] = []

for batch in tqdm(batches):
    try:
        resp = ollama.chat(
            model=OLLAMA_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": _make_user_prompt(batch)},
            ],
            options={"temperature": 0},
        )
        flagged_nodes.extend(_parse_response(resp["message"]["content"], batch))
    except Exception as exc:
        errors.append(f"{batch[0]['node_id']}: {exc}")

# ── Results ──────────────────────────────────────────────────────────────────
flagged_df = (
    pd.DataFrame(flagged_nodes)
    if flagged_nodes
    else pd.DataFrame(columns=["node_id", "node_type", "degree", "verdict", "reason"])
)

print(f"\nFlagged {len(flagged_df)} / {len(nodes_to_check)} nodes "
      f"({len(errors)} batch errors)")
if errors:
    print("Errors:", errors[:5])

print()
print(flagged_df.sort_values("degree", ascending=False).to_string(index=False))

Checking 3,296 nlp_ nodes (165 batches × 20)  model=qwen2.5:7b-instruct


  0%|          | 0/165 [00:00<?, ?it/s]

INFO httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: PO


Flagged 1603 / 3296 nodes (0 batch errors)

                                    node_id           node_type  degree   verdict                                                                                       reason
                               mode network             network    3008        NO                                                                               ambiguous term
                            stroke patients            disorder    2804        NO                                                                  patients are not a disorder
                           brain functional   anatomical_region    1881        NO                                                     functional is an adjective, not a region
                                 unaffected cognitive_construct    1732        NO                                                      unaffected is not a cognitive construct
                           controlled trial cognitive_construct    1708        N

In [20]:
# Remove all LLM-flagged nlp_ nodes from the canonical mapping.
# They will be treated as discarded — their edges are dropped during remapping.
# Re-run cells from "Step 4 — Master entity table" onward to propagate this change.

flagged_ids = set(flagged_df["node_id"])
n_before = len(nlp_to_canonical)
for nid in flagged_ids:
    nlp_to_canonical.pop(nid, None)

print(f"Removed  : {n_before - len(nlp_to_canonical):,} flagged nlp_ nodes")
print(f"Remaining: {len(nlp_to_canonical):,} canonical mappings")
print()
print("Re-run from 'Step 4 — Master entity table' to rebuild with updated mapping.")

Removed  : 0 flagged nlp_ nodes
Remaining: 3,971 canonical mappings

Re-run from 'Step 4 — Master entity table' to rebuild with updated mapping.


In [21]:
# Discarded terms — should be dominated by modality/method/metric generic words
discarded_terms = log_df[log_df["match_type"] == "discarded"]["entity"].tolist()
disc_nlp = nlp_nodes[nlp_nodes["node_id"].isin(discarded_terms)]

print("Discarded terms per node_type:")
print(disc_nlp["node_type"].value_counts().to_string())
print()
print("Top 20 discarded by degree (should be generic English words):")
print(disc_nlp.nlargest(20, "degree")[["node_id", "node_type", "degree"]].to_string(index=False))

Discarded terms per node_type:
node_type
modality             4208
method               1662
metric               1350
anatomical_region      47

Top 20 discarded by degree (should be generic English words):
         node_id         node_type  degree
          active          modality    9145
      prediction          modality    9083
  identification            method    8962
          direct          modality    8843
   statistically            method    8812
      anatomical            method    8783
      strategies            metric    8753
        physical            method    8642
       functions          modality    8297
  brain activity anatomical_region    8238
      predicting          modality    8151
       predicted          modality    7809
     interaction          modality    7682
        moderate            metric    7671
        spectrum          modality    7668
characterization          modality    7640
       detecting          modality    7586
    radiological  

## Step 4 — Master entity table

In [22]:
master_entities = build_master_entity_table(
    mesh_nodes, cogat_nodes, nlp_nodes, nlp_to_canonical
)

print(f"Total canonical entities: {len(master_entities):,}")
print()
print("Primary source breakdown:")
print(master_entities["primary_source"].value_counts().to_string())
print()
print("Multi-source nodes (>1 contributing KG):")
print(master_entities["sources"].value_counts().head(10).to_string())

INFO neurovlm.gnn.normalize: Master entity table: 33784 canonical entities


Total canonical entities: 33,784

Primary source breakdown:
primary_source
mesh        31110
nlp          1922
cogatlas      752

Multi-source nodes (>1 contributing KG):
sources
mesh                 29777
nlp                   1922
mesh,nlp              1182
cogatlas               680
cogatlas,mesh          110
cogatlas,nlp            72
cogatlas,mesh,nlp       41


## Step 5 — Remap edges to canonical IDs

In [23]:
# MeSH edges: identity mapping (UIs are already canonical)
mesh_id_map = {str(r["node_id"]): str(r["node_id"]) for _, r in mesh_nodes.iterrows()}
u_mesh = remap_edges(mesh_edges, mesh_id_map, drop_missing=False)
u_mesh["source_kg"] = "mesh"

# CogAtlas edges: already use canonical IDs from build_cogatlas_kg
u_cogat = pd.DataFrame()
if len(cogat_edges) > 0:
    cogat_id_map = {str(r["canonical_id"]): str(r["canonical_id"]) for _, r in cogat_nodes.iterrows()}
    u_cogat = remap_edges(cogat_edges, cogat_id_map, drop_missing=False)
    u_cogat["source_kg"] = "cogatlas"

# NLP edges: re-point NLP term → canonical ID, drop edges where either end is unmatched
u_nlp = remap_edges(nlp_edges, nlp_to_canonical, drop_missing=True)
u_nlp["source_kg"] = "nlp"

parts = [u_mesh, u_nlp]
if len(u_cogat) > 0:
    parts.append(u_cogat)

unified_edges = pd.concat(parts, ignore_index=True)
unified_edges = unified_edges.drop_duplicates(
    subset=["subject_id", "relation_type", "object_id", "source"]
)

print(f"MeSH edges          : {len(u_mesh):>8,}")
print(f"CogAtlas edges      : {len(u_cogat):>8,}")
print(f"NLP edges retained  : {len(u_nlp):>8,}  (of {len(nlp_edges):,} total)")
print(f"Unified edges total : {len(unified_edges):>8,}")
print()
print("Relation type breakdown:")
print(unified_edges["relation_type"].value_counts().to_string())

MeSH edges          :  305,227
NLP edges retained  :   24,339  (of 155,352 total)
Unified edges total :  329,566

Relation type breakdown:
relation_type
co_occurs_with              281629
narrower_term_of             42519
associated_with_disorder      2107
implicated_in                 1803
co_activates_with              931
expressed_in                   577


In [24]:
# Sanity check: all edge endpoints must be in the master entity table
canonical_ids = set(master_entities["canonical_id"])
sub_missing = ~unified_edges["subject_id"].isin(canonical_ids)
obj_missing = ~unified_edges["object_id"].isin(canonical_ids)
print(f"Edges with subject not in master table: {sub_missing.sum()}")
print(f"Edges with object not in master table : {obj_missing.sum()}")
assert sub_missing.sum() == 0 and obj_missing.sum() == 0, "Dangling edges — check remapping"

Edges with subject not in master table: 0
Edges with object not in master table : 0


## Save outputs

In [25]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

master_entities.to_parquet(OUT_DIR / "unified_kg_nodes.parquet", index=False)
unified_edges.to_parquet(OUT_DIR / "unified_kg_edges.parquet", index=False)
cogat_nodes.to_parquet(OUT_DIR / "cogat_kg_nodes.parquet", index=False)

cogatlas_merge_log = [
    {"entity": str(r["name"]), "trm_id": str(r["trm_id"]),
     "from_source": "cogatlas", "canonical_id": str(r["canonical_id"]),
     "match_type": str(r["match_type"])}
    for _, r in cogat_nodes.iterrows()
]
full_log = cogatlas_merge_log + nlp_merge_log
with open(OUT_DIR / "merge_log.json", "w") as fh:
    json.dump(full_log, fh, indent=2)

print(f"Saved {len(master_entities):,} nodes  → {OUT_DIR / 'unified_kg_nodes.parquet'}")
print(f"Saved {len(unified_edges):,} edges  → {OUT_DIR / 'unified_kg_edges.parquet'}")
print(f"Saved merge log ({len(full_log):,} records)  → {OUT_DIR / 'merge_log.json'}")

Saved 33,784 nodes  → data/unified_kg/unified_kg_nodes.parquet
Saved 329,566 edges  → data/unified_kg/unified_kg_edges.parquet
Saved merge log (13,759 records)  → data/unified_kg/merge_log.json


## Summary

In [26]:
log_df_all = pd.DataFrame(full_log)

print("=" * 50)
print("UNIFIED KG SUMMARY")
print("=" * 50)
print(f"  Canonical entities : {len(master_entities):>8,}")
print(f"    MeSH backbone    : {(master_entities['primary_source']=='mesh').sum():>8,}")
print(f"    CogAtlas new     : {(master_entities['primary_source']=='cogatlas').sum():>8,}")
print(f"    NLP new          : {(master_entities['primary_source']=='nlp').sum():>8,}")
print()
print(f"  Unified edges      : {len(unified_edges):>8,}")
print(f"    MeSH edges       : {len(u_mesh):>8,}")
print(f"    NLP co-occ edges : {len(u_nlp):>8,}")
print()
nlp_stats = log_df_all[log_df_all['from_source']=='nlp']['match_type'].value_counts()
print("  NLP match summary:")
for k, v in nlp_stats.items():
    print(f"    {k:<25s}: {v:>6,}")
print()
cogat_stats = log_df_all[log_df_all['from_source']=='cogatlas']['match_type'].value_counts()
print("  CogAtlas match summary:")
for k, v in cogat_stats.items():
    print(f"    {k:<25s}: {v:>6,}")

UNIFIED KG SUMMARY
  Canonical entities :   33,784
    MeSH backbone    :   31,110
    CogAtlas new     :      752
    NLP new          :    1,922

  Unified edges      :  329,566
    MeSH edges       :  305,227
    NLP co-occ edges :   24,339

  NLP match summary:
    discarded                :  7,267
    new_nlp_node             :  3,296
    mesh_synonym_match       :  1,662
    mesh_substring_match     :    543
    cogat_name_match         :     73

  CogAtlas match summary:
    new_cogat_node           :    752
    mesh_synonym_match       :    166


## Alternative: run everything via `run_normalization()`

Runs the full pipeline in one call. Equivalent to the cells above but without intermediate inspection.
Useful for re-running after tuning thresholds or updating `FORCE_MAP`.

In [ ]:
# results = run_normalization(
#     mesh_descriptors_df=mesh_descriptors,
#     mesh_nodes_df=mesh_nodes,
#     mesh_edges_df=mesh_edges,
#     nlp_nodes_df=nlp_nodes,
#     nlp_edges_df=nlp_edges,
#     cogatlas_out_path=COGAT_DIR / "cogatlas_concepts.parquet",
#     keep_unmatched_nlp=False,
#     keep_types=KEEP_TYPES,
#     out_dir=OUT_DIR,
# )
# results["master_entities"]["primary_source"].value_counts()